## Implementing `sqlitesearch` vector search in the RAG pipeline  

As previously discussed, we have already:

1. Retrieved our knowledge base.
2. Extracted the essential information needed to answer a question.
3. Concatenated that information into a list of strings.
4. Embedded this list of strings into a matrix.
5. Used `sqlitesearch.VectorSearchIndex` to create a persistent index (a .db file).
6. Called `fit` on the index using the embeddings (the matrix from the previous step) and the original text documents (from step 1).

Now, in a new Python session, we only need to re-import the embedder (to vectorize the user query) and instantiate a new index object. We do not need to call fit again, as the data has already been persisted to disk.

In [2]:
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex

model = SentenceTransformer("all-MiniLM-L6-v2")

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf", # k-means clustering
    db_path="zoomcamps.db" # use the persistent db file
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Set up the OpenAI client and create the assistant:

In [5]:
from rag_helper import RAGBase, RAGVector
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv() # load the environment variables from the .env file
openai_client = OpenAI()

In [ ]:
# Create a new RAG assistant object
vector_assistant = RAGVector(
    embedder=model,
    index=vs_index,
    llm_client=openai_client,
)

In [9]:
# Run a search when a user asks a question
answer = vector_assistant.rag_answer("the program has already begun, can I still sign up?")
answer

'Yes, you can still join. You can start learning and submitting homework while the form is open. If you want a certificate, make sure to submit your project before submissions close.'

In [10]:
# Close the connection to the index to avoid database corruption
vs_index.close()

# Comparing minsearch and sqlitesearch for vector search

- `minsearch VectorSearch`: in-memory (numpy), exact cosine similarity, must re-compute embeddings on startup, good for experiments and notebooks
- `sqlitesearch VectorSearchIndex`: persistent (SQLite .db file), ANN (LSH/IVF/HNSW) with exact rerank, can open an existing index, good for projects and persistence
This is probably the last you'll hear of sqlitesearch.
Alexey built it for teaching, to show the ingestion-then-deployment split.

It does have a real use though:
>Its only dependencies are `SQLite` and `numpy`.  
**So it runs on any host that offers a free SQLite database**, where a dedicated vector database would cost extra.  
For most work we may need something else, which is what we are going to do next.